In [ ]:
import sys
print(sys.executable)

### 1. Import data

In [1]:
import numpy as np
import pandas as pd
import duckdb

records = pd.read_parquet("../data/trials_australia.parquet")
print(f"All trials: {len(records):,}")

All trials: 12,642


### 2. One text per trial

In [2]:
trials = records.copy()
print(f"Trials in the index: {len(trials):,}")

def has(value) -> bool:
    """True if a value is present and not empty (handles None, NaN and pd.NA)."""
    return pd.notna(value) and str(value).strip() != ""

def trial_text(t) -> str:
    """The text that gets embedded: what the trial is about.
    Not included: phase, status, sponsor (those are filters)."""
    parts = [str(t["title"])] if has(t["title"]) else []
    if has(t["conditions"]):
        terms = f" ({t['condition_terms']})" if has(t["condition_terms"]) else ""
        parts.append(f"Conditions: {t['conditions']}{terms}")
    if has(t["interventions"]):
        parts.append(f"Interventions: {t['interventions']}")
    if has(t["summary"]):
        parts.append(str(t["summary"]))
    return ". ".join(p.strip().rstrip(".") for p in parts) + "."

trials["text"] = trials.apply(trial_text, axis=1)

words = trials["text"].str.split().str.len()
print(f"Words per text: median {words.median():.0f}, max {words.max()}")
print(f"Texts containing 'nan': {trials['text'].str.contains(r'\bnan\b', case=False).sum()}\n")
print(trials.loc[0, "text"])

Trials in the index: 12,642
Words per text: median 81, max 889
Texts containing 'nan': 0

Study to Evaluate Safety, Pharmacokinetics, and Efficacy of Rociletinib (CO-1686) in Previously Treated Mutant Epidermal Growth Factor Receptor (EGFR) in Non-Small Cell Lung Cancer (NSCLC) Patients. Conditions: Locally Advanced or Metastatic Non Small Cell Lung Cancer (Carcinoma, Non-Small-Cell Lung, Neoplasms, Neoplasm Metastasis, Recurrence). Interventions: Rociletinib. Rociletinib is a novel, potent, small molecule irreversible tyrosine kinase inhibitor (TKI) that selectively targets mutant forms of the epidermal growth factor receptor (EGFR) while sparing wild-type (WT) EGFR. The purpose of the study is to evaluate the pharmacokinetic (PK) and safety profile of oral rociletinib; to determine the maximum tolerated dose (MTD) and/or recommended Phase 2 dose (RP2D) of oral rociletinib; to assess the safety and efficacy of rociletinib in previously treated NSCLC patients known to have the T790M EG

### 3. Make sentence embedding

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")    # small, fast, 384 numbers per text

embeddings = model.encode(
    trials["text"].tolist(),
    batch_size=64,
    show_progress_bar=True,        # progress bar: batches done, time elapsed and time remaining
    normalize_embeddings=True,     # length 1: makes cosine similarity simple and reliable
)
print(f"Embeddings: {embeddings.shape}")   # (number of trials, 384)

W1112 21:31:39.900000 74383 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/198 [00:00<?, ?it/s]

Embeddings: (12642, 384)


### 4. Store trial + embeddings in DuckDB

In [9]:
DIM = embeddings.shape[1]
trials["embedding"] = list(embeddings.astype("float32"))

# Close an old connection if one is still open, then open a fresh one
try:
    con.close()
except NameError:
    pass
con = duckdb.connect("../data/pharma.duckdb")

# Register the pandas table under its own name, so DuckDB can't confuse it with the stored table
con.register("trials_df", trials)
con.execute(f"""
    CREATE OR REPLACE TABLE trials AS
    SELECT * EXCLUDE (embedding), embedding::FLOAT[{DIM}] AS embedding
    FROM trials_df
""")
con.unregister("trials_df")

# Checks
print("Trials in DuckDB:", con.execute("SELECT count(*) FROM trials").fetchone()[0])
print("Texts with 'Sponsor:':", con.execute("SELECT count(*) FROM trials WHERE text LIKE '%Sponsor:%'").fetchone()[0])

Trials in DuckDB: 12642
Texts with 'Sponsor:': 1


### 5. Test the search

In [10]:
def search(question: str, k: int = 10, phase: str | None = None,
           status: str | None = None, sponsor: str | None = None):
    """Most similar trials by meaning, with optional exact filters."""
    q = model.encode(question, normalize_embeddings=True).tolist()
    where, params = [], [q]
    if phase:
        where.append("phase = ?"); params.append(phase)
    if status:
        where.append("status = ?"); params.append(status)
    if sponsor:
        where.append("sponsor ILIKE ?"); params.append(f"%{sponsor}%")
    sql = f"""
        SELECT trial_id, title, conditions, interventions, phase, status, sponsor,
               round(array_cosine_similarity(embedding, ?::FLOAT[{DIM}]), 3) AS similarity,
               text
        FROM trials
        {"WHERE " + " AND ".join(where) if where else ""}
        ORDER BY similarity DESC
        LIMIT {k}
    """
    return con.execute(sql, params).df()

In [11]:
def show(results, n: int = 5):
    """Print the top results with the exact text that was embedded."""
    for i, r in results.head(n).iterrows():
        print(f"#{i + 1}  similarity {r.similarity}  |  {r.trial_id}  |  {r.phase}  |  {r.status}  |  {r.sponsor}")
        print(f"    {r.text[:600]}")
        print()

show(search("EGFR inhibitor for lung cancer"))

#1  similarity 0.7649999856948853  |  NCT02335944  |  Phase 1/2  |  TERMINATED  |  Novartis Pharmaceuticals
    Study of Safety and Efficacy of EGFR-TKI EGF816 in Combination With cMET Inhibitor INC280 in Adult Patients With EGFR Mutated Non Small Cell Lung Cancer. Conditions: Non Small Cell Lung Cancer (Carcinoma, Non-Small-Cell Lung). Interventions: Capmatinib, Nazartinib. The purpose of this study was to determine the maximum tolerated dose (MTD) / recommended phase 2 dose (RP2D) of nazartinib (EGF816) in combination with capmatinib (INC280) and to estimate the preliminary anti-tumor activity of nazartinib in combination with capmatinib in participants with advanced non-small cell lung cancer (NSCLC)

#2  similarity 0.7639999985694885  |  NCT01121575  |  Phase 1  |  COMPLETED  |  Pfizer
    A Study Of Combined C- MET Inhibitor And PAN-HER Inhibitor (PF-02341066 And PF-00299804) In Patients With Non- Small Cell Lung Cancer. Conditions: Non Small Cell Lung Cancer (Carcinoma, Non-Small

In [12]:
show(search("pembrolizumab"))                              # exact drug name

#1  similarity 0.597000002861023  |  NCT04303169  |  Phase 1/2  |  COMPLETED  |  Merck Sharp & Dohme LLC
    Substudy 02C: Safety and Efficacy of Pembrolizumab in Combination With Investigational Agents or Pembrolizumab Alone in Participants With Stage III Melanoma Who Are Candidates for Neoadjuvant Therapy (MK-3475-02C/KEYMAKER-U02). Conditions: Melanoma (Melanoma, Parkinson Disease 4, Autosomal Dominant Lewy Body). Interventions: Pembrolizumab, Vibostolimab, Gebasaxturev, MK-4830, Favezelimab + Pembrolizumab, ATRA. Substudy 02C is part of a larger research study that is testing experimental treatments for melanoma, a type of skin cancer. The larger study is the umbrella study.

The goal of substudy

#2  similarity 0.593999981880188  |  NCT03331731  |  Phase 2  |  COMPLETED  |  Peter MacCallum Cancer Centre, Australia
    A Phase II Study to Determine Pembrolizumab as Frontline Treatment of Patients With Hodgkin Lymphoma. Conditions: Hodgkin Lymphoma (Hodgkin Disease). Interventions: 

In [13]:
show(search("Keytruda"))                                   # brand name of pembrolizumab

#1  similarity 0.30000001192092896  |  NCT05280314  |  Phase 2  |  RECRUITING  |  IO Biotech
    Phase II Trial of Neoadjuvant and Adjuvant IO102-IO103 and Pembrolizumab KEYTRUDA® in Patients With Resectable Tumors. Conditions: Melanoma, Squamous Cell Carcinoma of Head and Neck (Melanoma, Squamous Cell Carcinoma of Head and Neck). Interventions: IO102-IO103, Pembrolizumab KEYTRUDA®. This is a multicenter, multi-arm trial evaluating anti-tumor activity, safety, and immune infiltration of IO102-IO103 in combination with pembrolizumab KEYTRUDA® as neoadjuvant and post-surgery treatment. This proof-of-concept trial will include patients with resectable tumors in at least 2 indications.

#2  similarity 0.25099998712539673  |  NCT07158918  |  Phase 1/2  |  RECRUITING  |  ABL Bio, Inc.
    ABL103 in Combination With Pembrolizumab, With or Without Taxane in Advanced or Metastatic Solid Tumors. Conditions: Solid Tumors. Interventions: ABL103, KEYTRUDA® (pembrolizumab), Taxane. This study is to 

In [14]:
show(search("Alzheimer's disease", status="RECRUITING"))   # with a filter

#1  similarity 0.5929999947547913  |  NCT07033494  |  Phase 2  |  RECRUITING  |  Merck Sharp & Dohme LLC
    A Clinical Study of MK-2214 in People With Early Alzheimer's Disease (MK-2214-004). Conditions: Early Alzheimer's Disease (Alzheimer Disease). Interventions: MK-2214, Placebo. Researchers want to know if the study treatment called MK-2214 works to slow certain changes in the brains of people with Alzheimer's disease (AD). AD is a type of dementia that can cause loss of memory, communication (such as speech), and decision-making skills. It can limit a person's ability to do daily tasks. MK-2214 is a study treatment designed to slow down AD.

The goals of the study are to learn:

* If MK-2214 sl

#2  similarity 0.5690000057220459  |  NCT07717411  |  Phase 3  |  RECRUITING  |  Hoffmann-La Roche
    A Study of Trontinemab in Cognitively Unimpaired Individuals at Risk for Progression to Symptomatic Alzheimer's Disease. Conditions: Alzheimer's Disease (Alzheimer Disease). Intervention

In [15]:
con.close()